In [ ]:
from pyspark.sql import functions as F

catalog = "case"
gold_schema = f"`{catalog}`.gold"
SEM_INFO = "Sem informação"

ft_chamados = spark.table(f"{gold_schema}.ft_chamados")
dim_motivo  = spark.table(f"{gold_schema}.dim_motivo")
dim_cliente = spark.table(f"{gold_schema}.dim_cliente")

print(f"ft_chamados: {ft_chamados.count():,} linhas")
print(f"dim_motivo:  {dim_motivo.count():,} linhas")
print(f"dim_cliente: {dim_cliente.count():,} linhas")

In [ ]:
base = (
    ft_chamados.alias("f")
    .join(dim_motivo.select("id_motivo", "categoria").alias("m"), on="id_motivo", how="left")
    .join(dim_cliente.select("id_cliente", "regiao", "faixa_etaria").alias("c"), on="id_cliente", how="left")
    .select(
        F.col("f.id_chamado"),
        F.col("id_cliente"),
        F.col("f.mes_abertura").alias("mes"),
        F.coalesce(F.col("m.categoria"), F.lit(SEM_INFO)).alias("categoria_motivo"),
        F.coalesce(F.col("c.regiao"), F.lit(SEM_INFO)).alias("regiao"),
        F.coalesce(F.col("c.faixa_etaria"), F.lit(SEM_INFO)).alias("faixa_etaria"),
        F.col("f.tempo_atendimento"),
    )
    .filter(F.col("mes").isNotNull())
)

display(base)

In [ ]:
GRAO = ["mes", "categoria_motivo", "regiao", "faixa_etaria"]

metricas = base.groupBy(*GRAO).agg(
    F.countDistinct("id_chamado").alias("qtd_chamados"),
    F.sum("tempo_atendimento").alias("soma_atendimento_seg"),
    F.count("tempo_atendimento").alias("qtd_atendimento_valido"),
    F.countDistinct("id_cliente").alias("qtd_clientes"),
)

total_mes = base.groupBy("mes").agg(F.countDistinct("id_chamado").alias("qtd_chamados_mes"))

kpi_motivos_clientes = (
    metricas.join(total_mes, on="mes", how="left")
    .select(
        *GRAO,
        "qtd_chamados",
        F.round(F.col("qtd_chamados") / F.col("qtd_chamados_mes") * 100, 2).alias("pct_volume"),
        F.round(F.col("soma_atendimento_seg") / F.col("qtd_atendimento_valido"), 1).alias("atendimento_medio_seg"),
        "qtd_clientes",
        F.round(F.col("qtd_chamados") / F.col("qtd_clientes"), 2).alias("chamados_por_cliente"),
        "soma_atendimento_seg",
        "qtd_atendimento_valido",
        "qtd_chamados_mes",
    )
)

kpi_motivos_clientes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.kpi_motivos_clientes")

# Total geral de chamados, usado nos percentuais das análises abaixo
total = kpi_motivos_clientes.agg(F.sum("qtd_chamados")).first()[0]


display(kpi_motivos_clientes.orderBy("mes", F.desc("qtd_chamados")))

### Faixa etarias com mais chamados

In [ ]:
clientes_por_faixa = (
    spark.table("`case`.gold.dim_cliente")
    .groupBy("faixa_etaria")
    .agg(F.count("*").alias("qtd_clientes_cadastrados"))
)

# Faixa etaria que tem mais chamados por pessoa
display(
    kpi_motivos_clientes.groupBy("faixa_etaria")
    .agg(F.sum("qtd_chamados").alias("qtd_chamados"))
    .join(clientes_por_faixa, on="faixa_etaria", how="left")
    .withColumn("pct_chamados", F.round(F.col("qtd_chamados") / total * 100, 2))
    .withColumn("chamados_por_cliente", F.round(F.col("qtd_chamados") / F.col("qtd_clientes_cadastrados"), 2))
    .orderBy(F.desc("chamados_por_cliente"))
)

### Regiao com mais chamados

In [ ]:
clientes_por_regiao = (
    spark.table("`case`.gold.dim_cliente")
    .groupBy("regiao")
    .agg(F.count("*").alias("qtd_clientes_cadastrados"))
)

display(
    kpi_motivos_clientes.groupBy("regiao")
    .agg(F.sum("qtd_chamados").alias("qtd_chamados"))
    .join(clientes_por_regiao, on="regiao", how="left")
    .withColumn("pct_chamados", F.round(F.col("qtd_chamados") / total * 100, 2))
    .withColumn("chamados_por_cliente", F.round(F.col("qtd_chamados") / F.col("qtd_clientes_cadastrados"), 2))
    .orderBy(F.desc("chamados_por_cliente"))
)

### motivo mais chamado

In [ ]:
display(
    kpi_motivos_clientes.groupBy("categoria_motivo")
    .agg(
        F.sum("qtd_chamados").alias("qtd_chamados"),
        F.round(F.sum("soma_atendimento_seg") / F.sum("qtd_atendimento_valido"), 1).alias("atendimento_medio_seg"),
    )
    .withColumn("pct_chamados", F.round(F.col("qtd_chamados") / total * 100, 2))
    .orderBy(F.desc("qtd_chamados"))
)

### Motivos mais chamados por idade

In [ ]:
qtd = kpi_motivos_clientes.groupBy("faixa_etaria", "categoria_motivo").agg(F.sum("qtd_chamados").alias("qtd"))
qtd_faixa = kpi_motivos_clientes.groupBy("faixa_etaria").agg(F.sum("qtd_chamados").alias("qtd_faixa"))

# % de cada categoria dentro de cada faixa etária
pct_faixa = (
    qtd.join(qtd_faixa, "faixa_etaria")
    .withColumn("pct", F.round(F.col("qtd") / F.col("qtd_faixa") * 100, 2))
    .select("faixa_etaria", "categoria_motivo", "pct")
)

# Linha de referência: % de cada categoria na base toda
pct_total = (
    kpi_motivos_clientes.groupBy("categoria_motivo")
    .agg(F.round(F.sum("qtd_chamados") / total * 100, 2).alias("pct"))
    .withColumn("faixa_etaria", F.lit("total"))
    .select("faixa_etaria", "categoria_motivo", "pct")
)

display(
    pct_faixa.unionByName(pct_total)
    .groupBy("faixa_etaria").pivot("categoria_motivo").agg(F.first("pct"))
    .orderBy("faixa_etaria")
)

### Motivos mais chamados por região

In [ ]:
qtd = kpi_motivos_clientes.groupBy("regiao", "categoria_motivo").agg(F.sum("qtd_chamados").alias("qtd"))
qtd_faixa = kpi_motivos_clientes.groupBy("regiao").agg(F.sum("qtd_chamados").alias("qtd_faixa"))

# % de cada categoria dentro de cada região
pct_faixa = (
    qtd.join(qtd_faixa, "regiao")
    .withColumn("pct", F.round(F.col("qtd") / F.col("qtd_faixa") * 100, 2))
    .select("regiao", "categoria_motivo", "pct")
)

# Linha de referência: % de cada categoria na base toda
pct_total = (
    kpi_motivos_clientes.groupBy("categoria_motivo")
    .agg(F.round(F.sum("qtd_chamados") / total * 100, 2).alias("pct"))
    .withColumn("regiao", F.lit("total"))
    .select("regiao", "categoria_motivo", "pct")
)

display(
    pct_faixa.unionByName(pct_total)
    .groupBy("regiao").pivot("categoria_motivo").agg(F.first("pct"))
    .orderBy("regiao")
)

### chamados por cliente

In [ ]:
clientes_com_chamado = (
    ft_chamados.filter(F.col("id_cliente").isNotNull())
    .select("id_cliente").distinct()
    .withColumn("abriu_chamado", F.lit(1))
)

display(
    dim_cliente.join(clientes_com_chamado, on="id_cliente", how="left")
    .groupBy("faixa_etaria")
    .agg(
        F.count("*").alias("qtd_clientes_cadastrados"),
        F.sum(F.coalesce(F.col("abriu_chamado"), F.lit(0))).alias("qtd_clientes_impactados"),
    )
    .withColumn("pct_clientes_impactados",
                F.round(F.col("qtd_clientes_impactados") / F.col("qtd_clientes_cadastrados") * 100, 2))
    .orderBy(F.desc("pct_clientes_impactados"))
)